# Q02 — High Earners in Support Departments (Amazon)

**Python for Data Science — Company Interview Question Bank**

| Field | Detail |
|---|---|
| Question ID | `PANDAS-02-Q002` |
| Primary category | 02 — Indexing, Selection and Filtering |
| Company association | Amazon (StrataScratch listing; not an independently authenticated interview transcript) |
| Position category | Data Scientist (StrataScratch Amazon interview-prep track) |
| Original source difficulty | Easy |
| Extended depth | Intermediate–Advanced (original educational extensions) |
| Original question ID | StrataScratch #2167 |
| Key concepts | Boolean masks, `&` / `\|`, `.isin()`, `.loc[]`, column projection, thresholds, label alignment, nullable Boolean values |
| Original problem | [StrataScratch — High Earners in Support Departments](https://platform.stratascratch.com/coding/2167-high-earners-in-support-departments) |
| Interview-prep track | [StrataScratch — Amazon Data Scientist](https://platform.stratascratch.com/interview-loops/amazon/data-scientist) |

> **Authenticity boundary:** StrataScratch publicly lists a problem with this title, attributes it to Amazon among its company tags, and gives the task of selecting HR/Admin employees who earn more than $80,000. The visible record is a *question-bank attribution*, **not independent proof that this exact wording appeared in an actual Amazon interview**. The table, identifiers, Pandas implementation, tests, extensions and follow-up questions below are original educational adaptations. The source platform primarily presents the task in SQL.

## 1. Interview Question

**Company-associated problem (paraphrased):**

An HR team is conducting a salary-benchmarking study. Identify employees who meet **both** conditions:

1. Their annual salary is **strictly greater than $80,000**.
2. Their department is **HR** or **Admin**.

Return these four columns in exactly this order: `first_name`, `last_name`, `department`, `salary`.

### Pandas-specific rules for this exercise

- Input: a DataFrame named `employees`, indexed by `employee_id`.
- Keep the input row order and original index labels; do not reset or sort.
- Do not include `employee_id` as a data column or include unrelated fields in the returned DataFrame.
- Treat missing salaries or departments as *not qualifying*; do not fill in missing salary values.
- Do not mutate `employees`.
- Use vectorized Pandas filtering, not a Python row loop.

**Important:** The index, null-handling policy and output-order conventions above are educational additions, not verified original-company requirements.

## 2. Mathematical Definition

Let employee $i$ have salary $s_i$ and department $d_i$. Let $D=\{\mathrm{HR},\mathrm{Admin}\}$ and threshold $\tau=80{,}000$.

The selection predicate is

$$
\operatorname{keep}(i)=(s_i>\tau)\;\land\;(d_i\in D).
$$

The salary inequality is **strict**: $s_i=80{,}000$ does **not** qualify.

For a record with an unknown salary or department, the exercise treats the predicate as false. Missing data should never be silently imputed solely to pass the filter.

## 3. Setup and Synthetic Input Data

The following table is synthetic; its role is to make the public problem statement executable and to test boundary conditions. Nonconsecutive `employee_id` labels are intentional.

In [1]:
import math
import numbers
import numpy as np
import pandas as pd

print("Pandas version:", pd.__version__)

records = [
    {"employee_id": 101, "first_name": "Ava",    "last_name": "Nguyen", "department": "HR",          "salary":  95_000.00, "office": "NY"},
    {"employee_id": 102, "first_name": "Ben",    "last_name": "Carter", "department": "Admin",       "salary":  80_000.00, "office": "CA"},
    {"employee_id": 103, "first_name": "Chloe",  "last_name": "Patel",  "department": "Engineering", "salary": 140_000.00, "office": "NY"},
    {"employee_id": 104, "first_name": "Daniel", "last_name": "Ross",   "department": "Admin",       "salary":  82_000.00, "office": "MA"},
    {"employee_id": 105, "first_name": "Eva",    "last_name": "Li",     "department": "HR",          "salary":  80_001.00, "office": "NY"},
    {"employee_id": 106, "first_name": "Farah",  "last_name": "Kim",    "department": "Sales",       "salary": 105_000.00, "office": "TX"},
    {"employee_id": 107, "first_name": "Gabe",   "last_name": "Park",   "department": "HR",          "salary":  75_000.00, "office": "NY"},
    {"employee_id": 108, "first_name": "Hana",   "last_name": "Perez",  "department": "Admin",       "salary": 125_000.00, "office": "CA"},
    {"employee_id": 109, "first_name": "Ivan",   "last_name": "Reed",   "department": None,          "salary":  99_000.00, "office": "NY"},
    {"employee_id": 110, "first_name": "June",   "last_name": "Moss",   "department": "HR",          "salary":       None, "office": "MA"},
    {"employee_id": 111, "first_name": "Kai",    "last_name": "Shah",   "department": "Finance",     "salary": 100_000.00, "office": "NY"},
    {"employee_id": 112, "first_name": "Lina",   "last_name": "Chen",   "department": "Admin",       "salary":  80_000.00, "office": "CA"},
    {"employee_id": 113, "first_name": "Mina",   "last_name": "Cole",   "department": "HR",          "salary": 160_000.00, "office": "MA"},
    {"employee_id": 114, "first_name": "Noah",   "last_name": "Lee",    "department": "HR",          "salary":  80_000.01, "office": "TX"},
    {"employee_id": 115, "first_name": "Omar",   "last_name": "West",   "department": "Admin",       "salary":       None, "office": "CA"},
]

employees = pd.DataFrame.from_records(records).set_index("employee_id")
employees.index.name = "employee_id"

OUTPUT_COLUMNS = ["first_name", "last_name", "department", "salary"]
SUPPORT_DEPARTMENTS = ("HR", "Admin")
SALARY_THRESHOLD = 80_000

print(employees.to_string())
print("\nDtypes:\n", employees.dtypes)
assert employees.index.tolist() == list(range(101, 116))

Pandas version: 2.3.3
            first_name last_name   department     salary office
employee_id                                                    
101                Ava    Nguyen           HR   95000.00     NY
102                Ben    Carter        Admin   80000.00     CA
103              Chloe     Patel  Engineering  140000.00     NY
104             Daniel      Ross        Admin   82000.00     MA
105                Eva        Li           HR   80001.00     NY
106              Farah       Kim        Sales  105000.00     TX
107               Gabe      Park           HR   75000.00     NY
108               Hana     Perez        Admin  125000.00     CA
109               Ivan      Reed         None   99000.00     NY
110               June      Moss           HR        NaN     MA
111                Kai      Shah      Finance  100000.00     NY
112               Lina      Chen        Admin   80000.00     CA
113               Mina      Cole           HR  160000.00     MA
114               

## 4. Standard Pandas Solution — Two Conditions and One Selection

Use `.gt()` for a strict salary comparison, `.isin()` for set membership, and the **elementwise** `&` operator to combine Boolean Series. `.loc` then filters rows and projects the required columns.

**Do not use** Python's scalar `and` to combine Series. Each comparison should be parenthesized if written inline with `&` or `|`.

In [2]:
# A clear interview-ready solution.
salary_condition = employees["salary"].gt(SALARY_THRESHOLD).fillna(False)
department_condition = employees["department"].isin(SUPPORT_DEPARTMENTS)

selection_mask = salary_condition & department_condition
result = employees.loc[selection_mask, OUTPUT_COLUMNS].copy()

print("Salary condition:\n", salary_condition.to_string())
print("\nDepartment condition:\n", department_condition.to_string())
print("\nCombined mask:\n", selection_mask.to_string())
print("\nFINAL RESULT:\n", result.to_string())

Salary condition:
 employee_id
101     True
102    False
103     True
104     True
105     True
106     True
107    False
108     True
109     True
110    False
111     True
112    False
113     True
114     True
115    False

Department condition:
 employee_id
101     True
102     True
103    False
104     True
105     True
106    False
107     True
108     True
109    False
110     True
111    False
112     True
113     True
114     True
115     True

Combined mask:
 employee_id
101     True
102    False
103    False
104     True
105     True
106    False
107    False
108     True
109    False
110    False
111    False
112    False
113     True
114     True
115    False

FINAL RESULT:
             first_name last_name department     salary
employee_id                                           
101                Ava    Nguyen         HR   95000.00
104             Daniel      Ross      Admin   82000.00
105                Eva        Li         HR   80001.00
108               Hana     P

## 5. Expected Output and Boundary Interpretation

Exactly six employees qualify. Their row indices are **101, 104, 105, 108, 113, 114**, in that order.

| employee_id (index) | first_name | last_name | department | salary |
|---:|---|---|---|---:|
| 101 | Ava | Nguyen | HR | 95000.00 |
| 104 | Daniel | Ross | Admin | 82000.00 |
| 105 | Eva | Li | HR | 80001.00 |
| 108 | Hana | Perez | Admin | 125000.00 |
| 113 | Mina | Cole | HR | 160000.00 |
| 114 | Noah | Lee | HR | 80000.01 |

**Excluded deliberately:** employee 102 and 112 have salary exactly 80000; 103/106/111 have sufficiently high salaries but incorrect departments; 107 is in HR but below threshold; 109/110/115 contain relevant missing values.

## 6. Automated Verification — Correctness, Schema, and No Mutation

Do not validate only printed rows: confirm the exact index, exact output columns, strict boundary rule, preserved order, and unchanged input.

In [3]:
expected_ids = [101, 104, 105, 108, 113, 114]
expected = employees.loc[expected_ids, OUTPUT_COLUMNS].copy()

pd.testing.assert_frame_equal(result, expected)
assert result.index.tolist() == expected_ids
assert result.index.name == "employee_id"
assert result.columns.tolist() == OUTPUT_COLUMNS
assert (result["salary"] > 80_000).all()
assert result["department"].isin(["HR", "Admin"]).all()
assert 102 not in result.index and 112 not in result.index
assert 109 not in result.index and 110 not in result.index

original_snapshot = employees.copy(deep=True)
_ = employees.loc[selection_mask, OUTPUT_COLUMNS].copy()
pd.testing.assert_frame_equal(employees, original_snapshot)
print("All Q02 core tests passed.")

All Q02 core tests passed.


## 7. Deep Dive — Boolean Operators, Precedence and `.isin()`

### 7.1 How `&`, `|`, and `~` work

- `&` is elementwise logical **AND**.
- `|` is elementwise logical **OR**.
- `~` is elementwise logical **NOT** (on Boolean masks).
- Python `and`, `or`, and `not` are **not interchangeable** with these elementwise operators.

The core predicate can also be written as:

```python
mask = (
    (employees["salary"] > 80_000)
    & (
        (employees["department"] == "HR")
        | (employees["department"] == "Admin")
    )
)
```

**Avoid:** `employees["salary"] > 80_000 & employees["department"].isin(...)` because the missing parentheses make the expression parse incorrectly. In general, always parenthesize each comparison around `&` and `|`.

### 7.2 Why `.isin()`?

`department.isin(["HR", "Admin"])` expresses set membership, avoiding repeated equality checks. It matches **exact values**, not case-insensitive variants or arbitrary whitespace.

In [4]:
# Show that multiple equality comparisons are equivalent to .isin() here.
explicit_department_condition = (
    employees["department"].eq("HR")
    | employees["department"].eq("Admin")
)

pd.testing.assert_series_equal(department_condition, explicit_department_condition)

# Elementwise complement: rows outside the target departments.
not_support = ~department_condition
assert not_support.loc[103] and not_support.loc[106]
assert not not_support.loc[101]
print(".isin(), elementwise OR, and Boolean NOT behave as expected.")

.isin(), elementwise OR, and Boolean NOT behave as expected.


## 8. `.loc` vs `.iloc`: Labels, Positions and Mask Alignment

`selection_mask` is a **Series with index labels**. For a DataFrame with a unique index, `.loc` can align a Boolean Series by those labels even if the Series is in a different order.

An `.iloc` Boolean indexer, by contrast, must be a positional Boolean array (for example `mask.to_numpy()`). Converting to NumPy discards the labels; this is correct **only when the array is already in row order**.

The next example deliberately reverses the mask to expose the difference.

In [5]:
reordered_mask = selection_mask.iloc[::-1]

label_based_result = employees.loc[reordered_mask, OUTPUT_COLUMNS]
pd.testing.assert_frame_equal(label_based_result, expected)

# A NumPy mask is interpreted positionally and cannot align by employee_id.
position_based_result = employees.iloc[
    reordered_mask.to_numpy(dtype=bool),
    employees.columns.get_indexer(OUTPUT_COLUMNS),
]

print("Correct label-aligned index:", label_based_result.index.tolist())
print("Wrong positional index from the reversed mask:", position_based_result.index.tolist())
assert position_based_result.index.tolist() != expected_ids

Correct label-aligned index: [101, 104, 105, 108, 113, 114]
Wrong positional index from the reversed mask: [102, 103, 108, 111, 112, 115]


## 9. Robust Reusable Implementation

For production pipelines, validate the required columns, reject duplicate column labels, validate salary dtype, and make the threshold and department set configurable.

This version deliberately uses **exact department names**. Case folding, whitespace trimming and currency conversion are separate data-cleaning policies and should not be introduced without authorization.

In [6]:
def find_high_earning_support_employees(
    frame: pd.DataFrame,
    min_salary: float = 80_000,
    departments: tuple[str, ...] = ("HR", "Admin"),
) -> pd.DataFrame:
    """Filter employees above a strict salary threshold in selected departments.

    Keeps the input's row order/index and projects exactly four output columns.
    Missing salaries/departments do not qualify. Does not mutate the input.
    """
    required = ["first_name", "last_name", "department", "salary"]

    if not isinstance(frame, pd.DataFrame):
        raise TypeError("frame must be a pandas DataFrame")
    if not frame.columns.is_unique:
        raise ValueError("Column labels must be unique")

    absent = [col for col in required if col not in frame.columns]
    if absent:
        raise KeyError(f"Missing required columns: {absent}")

    if (not isinstance(min_salary, numbers.Real)
            or isinstance(min_salary, bool)
            or not math.isfinite(min_salary)):
        raise ValueError("min_salary must be a finite real number")
        
    if isinstance(departments, (str, bytes)) or not departments:
        raise ValueError("departments must be a nonempty sequence of labels")
    if not pd.api.types.is_numeric_dtype(frame["salary"]):
        raise TypeError("salary must have a numeric dtype")

    above_threshold = frame["salary"].gt(min_salary).fillna(False)
    allowed_department = frame["department"].isin(departments)
    keep = above_threshold & allowed_department

    return frame.loc[keep, required].copy()

filtered = find_high_earning_support_employees(employees)
pd.testing.assert_frame_equal(filtered, expected)
print(filtered.to_string())

            first_name last_name department     salary
employee_id                                           
101                Ava    Nguyen         HR   95000.00
104             Daniel      Ross      Admin   82000.00
105                Eva        Li         HR   80001.00
108               Hana     Perez      Admin  125000.00
113               Mina      Cole         HR  160000.00
114               Noah       Lee         HR   80000.01


## 10. Edge Cases and Robustness Tests

Test important edge cases rather than assuming the main example covers them:

1. **Exclusive threshold:** 80000 does not pass, but 80000.01 passes.
2. **Missing values:** nullable salary and missing department are excluded.
3. **Extra columns:** not included in the output.
4. **Empty DataFrame:** returns an empty DataFrame with the expected four columns and preserved index metadata.
5. **Duplicate *row* indices:** allowed when the mask is computed from the same frame; a row's identity is not necessarily a unique label.
6. **Duplicate *column* labels:** rejected by this reusable function.
7. **Missing required column / invalid dtype / invalid parameter:** a clear exception is raised.

In [7]:
# Core boundary checks.
assert 114 in filtered.index  # 80000.01 > 80000
assert 102 not in filtered.index  # 80000 is not > 80000
assert 112 not in filtered.index

# Test nullable Pandas extension dtype and pd.NA.
nullable = employees.copy()
nullable["salary"] = nullable["salary"].astype("Float64")
nullable.loc[110, "salary"] = pd.NA
nullable.loc[109, "department"] = pd.NA
nullable_result = find_high_earning_support_employees(nullable)
assert nullable_result.index.tolist() == expected_ids

# Empty input with a valid schema.
empty_result = find_high_earning_support_employees(employees.iloc[0:0])
assert empty_result.empty
assert empty_result.columns.tolist() == OUTPUT_COLUMNS
assert empty_result.index.name == "employee_id"

# Equal row labels are not necessarily equal observations.
with_duplicate_row_index = employees.copy()
with_duplicate_row_index.index = pd.Index([1, 1] + list(range(3, 16)), name="employee_id")
with_duplicate_result = find_high_earning_support_employees(with_duplicate_row_index)
assert with_duplicate_result.iloc[0]["first_name"] == "Ava"
assert len(with_duplicate_result) == 6

print("Boundary, nullable, empty-input and duplicate-row-index tests passed.")

Boundary, nullable, empty-input and duplicate-row-index tests passed.


In [8]:
# Validate failure modes and input immutability.
def check_raises(expected_exception, invalid_input, **kwargs):
    try:
        find_high_earning_support_employees(invalid_input, **kwargs)
    except expected_exception:
        return
    raise AssertionError(f"Expected {expected_exception.__name__}")

check_raises(KeyError, employees.drop(columns="department"))
check_raises(TypeError, employees.assign(salary=employees["salary"].astype(str)))
check_raises(ValueError, employees, departments=())
check_raises(ValueError, employees, min_salary=float("nan"))
check_raises(ValueError, employees, min_salary=float("inf"))

bad_columns = employees.copy()
bad_columns.columns = ["first_name", "last_name", "department", "salary", "salary"]
check_raises(ValueError, bad_columns)

before = employees.copy(deep=True)
_ = find_high_earning_support_employees(employees)
pd.testing.assert_frame_equal(employees, before)
print("All input validation and non-mutation tests passed.")

All input validation and non-mutation tests passed.


## 11. Alternative Implementations

### 11.1 One expression with `.loc`

A concise implementation is often enough for an interview:

```python
answer = employees.loc[
    (employees["salary"] > 80_000)
    & employees["department"].isin(["HR", "Admin"]),
    ["first_name", "last_name", "department", "salary"],
].copy()
```

### 11.2 `DataFrame.query`

`.query()` can improve readability for simple predicates but introduces a string-expression language and potential quoting issues for column names. When filtering untrusted or user-provided inputs, do not concatenate raw values into a query string; reference Python variables with `@` where appropriate. `engine="python"` is explicit here for compatibility with a wider range of dtypes.

### 11.3 `.between()` is **not** the same condition

`salary.between(80_000, 160_000)` is **inclusive** by default and includes 80000. That is incorrect for this question unless one explicitly changes the left bound behavior. An upper bound is not needed at all in the original task.

In [9]:
compact = employees.loc[
    (employees["salary"] > SALARY_THRESHOLD)
    & employees["department"].isin(SUPPORT_DEPARTMENTS),
    OUTPUT_COLUMNS,
].copy()
pd.testing.assert_frame_equal(compact, result)

threshold = SALARY_THRESHOLD
departments = list(SUPPORT_DEPARTMENTS)
queried = employees.query(
    "salary > @threshold and department in @departments",
    engine="python",
).loc[:, OUTPUT_COLUMNS].copy()
pd.testing.assert_frame_equal(queried, result)

# Demonstrate why .between() with default inclusive bounds is not suitable.
between_mask = employees["salary"].between(80_000, 160_000)
assert bool(between_mask.loc[102])  # incorrectly includes salary=80000
print("Inline .loc and query implementations match; between() boundary caveat verified.")

Inline .loc and query implementations match; between() boundary caveat verified.


## 12. Time and Space Complexity

Let $n$ be the number of employees, and let $r$ be the number of qualifying rows. Assume a fixed number of selected columns and a small, fixed department set.

| Operation | Typical time | Additional memory |
|---|---|---|
| Compare salary against threshold | $O(n)$ | $O(n)$ Boolean mask |
| Department membership (`isin`) | $O(n)$ for fixed small set | $O(n)$ mask; small membership structure |
| Combine the masks | $O(n)$ | $O(n)$ combined mask |
| Filter and copy four columns | $O(r)$ for fixed width | $O(r)$ output data (plus index) |

Consequently, the typical model is

$$
T(n,r)=O(n+r)=O(n),\qquad M(n,r)=O(n+r)=O(n).
$$

This is a **model**, not a guarantee for all object types, exotic indexes or Pandas backends. String comparisons, extension dtypes, references and memory layout can change constants and allocations. If department-set cardinality $d$ is not fixed, building a membership structure can add $O(d)$ cost.

**Vectorized** does not mean zero intermediate memory: each predicate normally materializes or otherwise manages Boolean data.

## 13. Common Interview Pitfalls

1. **Confusing `>` with `>=`:** salary exactly 80000 must be excluded.
2. **Using `and` instead of `&`:** Python `and` cannot combine non-scalar Pandas Boolean Series.
3. **Misplacing parentheses:** comparisons must be grouped before applying `&` and `|`.
4. **Using `|` between the salary and department conditions:** would return employees satisfying only one criterion.
5. **Dropping index labels:** `reset_index(drop=True)` violates the requirements in this adaptation.
6. **Returning the entire table:** the output is exactly four requested columns.
7. **Ignoring missing values:** do not accidentally let null salaries or departments pass.
8. **Case and whitespace normalization without specification:** `HR`, `hr` and `HR ` are different strings.
9. **Assuming `.iloc` aligns Series by labels:** it is positional.
10. **Mutating the caller's data:** use a clearly independent output for subsequent changes.

## 14. Advanced Follow-Up Questions — Original Educational Extensions

**Q1. How would you extend the condition to `(salary > 80000) AND (department in HR/Admin) AND (office != NY)`?**  
Combine three parenthesized Boolean masks using `&`, with the exclusion condition using `.ne("NY")`. Decide explicitly what missing office values should mean.

**Q2. How does `.isin()` differ from repeated `==` comparisons?**  
`.isin()` directly represents membership in a collection and is easier to extend; repeated equality masks combined with `|` are equivalent for the exact strings used here.

**Q3. Why can a reversed Boolean Series still work with `.loc`?**  
With unique and matching labels, `.loc` aligns by label. `.iloc` with a raw Boolean array instead applies positions.

**Q4. Why does salary equal to 80000 fail this filter?**  
Because the threshold comparison is strictly `>`, not `>=`.

**Q5. What if salary is stored as strings such as `"$95,000"`?**  
It is a data-cleaning/schema problem: explicitly parse currency and separators, validate conversion, and decide how to treat invalid observations before filtering. The production-style function intentionally rejects a nonnumeric salary column.

**Q6. How do nullable Pandas Boolean values affect filtering?**  
A comparison on nullable values can produce `pd.NA`. Explicit `.fillna(False)` makes the exclusion policy clear; do not silently treat unknown observations as valid.

**Q7. Should you drop employees with missing salaries before performing compensation analysis?**  
Only if the missing-data mechanism, study population and downstream inferential consequences have been considered; filtering and imputing can introduce selection bias.

**Q8. How would you adapt the predicate to a Quant universe selection task?**  
For example, `(market_cap > minimum) & sector.isin(allowed_sectors)` with timestamp and instrument identity preserved. Include liquidity and as-of-date policies to avoid look-ahead bias.

## 15. Analytical Interpretation

The result identifies an **eligible subset**, not evidence of a pay disparity or a causal difference between HR and Admin. Since salary is the selection variable, any average salary calculated on this subset is truncated from below; it is *not* the overall salary mean for either department.

**Selection bias / sample definition:** A dataset containing only employees above a threshold cannot directly estimate the full compensation distribution. Additionally, employees with missing salaries or unknown departments are excluded by policy; missingness could be associated with seniority, location, or employment type.

**Index integrity:** Preserving `employee_id` allows the selected records to be traced back to source systems and joined safely with other employee-level data. Changing to a positional mask built from a different row order can silently select the wrong employees.

**Quant analogy:** Financial universe filters (e.g., `volume > threshold` AND `sector in allowed_set`) require equally explicit rules about thresholds, missing observations, and the timestamp at which each eligibility criterion became known.

Strong candidates articulate the precise predicate, null policy, index semantics, and the population to which any subsequent statistic applies.

## 16. Concise Interview-Ready Summary

> I construct a salary mask with `salary > 80000`, a department-membership mask with `department.isin(["HR", "Admin"])`, then combine them using elementwise `&`. I apply the Boolean mask with `.loc` and select only the four required columns. The threshold is strict, missing relevant fields do not qualify, and the input index and row order are preserved. For a fixed number of output columns, the filtering step is typically linear in the number of rows.

**Completion checklist:**

- [ ] Predict the six matching `employee_id` labels without running code.
- [ ] Explain why salaries exactly equal to 80000 are excluded.
- [ ] Explain why `&` works but Python `and` does not.
- [ ] Explain `.isin()` and why parentheses matter.
- [ ] Explain the difference between label-aligned `.loc` and positional `.iloc`.
- [ ] Run all tests, including missing values, invalid inputs and no mutation.
- [ ] Explain why the selected sample cannot represent the full salary distribution.

**Source:** [StrataScratch, ID #2167](https://platform.stratascratch.com/coding/2167-high-earners-in-support-departments). All examples, explanatory code, and expanded challenges here are original educational material.